# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Detección de Argumentos con deepseek-r1:70b en poliGPT API

- ollama serve
- ollama run gemma3:4b

In [ ]:
#%pip install langchain pymupdf openai openpyxl pandas numpy openpyxl --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [21]:
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
import requests
import json
import re
from openai import OpenAI
import openai
import httpx
import pandas as pd
import numpy as np
import os
import openpyxl

process_text_path = "..\\Data\\Processed Files (sections)\\"

model_name="deepseek-r1:70b"
prefix = 'GLOBAL_SGD2023'
output_dir = "..\\Data\\Extracted Arguments No Keywords (all text)\\"

## Input text processing

In [ ]:
# 1. Define your Pydantic schema for output
class ArgumentResponse(BaseModel):
    arguments: List[str] = Field(..., description="List of arguments extracted directly from the text.")

# 2. Setup output parser
pydantic_parser = PydanticOutputParser(pydantic_object=ArgumentResponse)

# 3. Extend text with first sentence from the next page
def extend_pages_with_next_sentence(pages):
    def get_first_sentence(text):
        match = re.search(r'(.+?\.)', text.strip())
        return match.group(1).strip() if match else ""

    extended_pages = []
    for i, page in enumerate(pages):
        current_text = page["text"]
        if i + 1 < len(pages):
            next_sentence = get_first_sentence(pages[i + 1]["text"])
            current_text += " " + next_sentence
        extended_pages.append({
            "page": page["page"],
            "text": current_text
        })
    return extended_pages

# 4. Build the prompt and call the LLM to extract arguments
def extract_arguments_json(text, topic, model_name) -> ArgumentResponse:
    format_instructions = pydantic_parser.get_format_instructions()

    prompt = PromptTemplate(
        template=(
            "Task: Text Span Identification for Arguments related to Sustainable Development Goal: {topic}\n\n"
            "Role: You are an expert in logical reasoning, sustainability reporting, and argument analysis. "
            "Your job is to identify and extract **verbatim arguments** about {topic} from long-form sustainability texts.\n\n"
            "Instructions:\n"
            "1. Carefully read the entire input text.\n"
            "2. Identify ONLY those sentences or phrases that:\n"
            "   - Clearly support or argue for or against the topic {topic}\n"
            "   - Contain keyword from the relevant lists below\n"
            "   - Are exclusively about {topic} (EXCLUDE if they mention or refer to other SDGs or unrelated sustainability topics)\n\n"
            "3. Each extracted argument must:\n"
            "   - Relate exclusively to the specified SDG ({topic})\n"
            "   - Stand as a full statement\n"
            "   - Be copied exactly from the original (no paraphrasing)\n"
            "   - Include only the necessary context for understanding\n"
            "4. If no qualifying arguments are found, return an empty array.\n\n"
            "Output Rules:\n"
            "- Use **only the exact text** from the original\n"
            "- Do **not** add or reword anything\n"
            "- Return only valid JSON\n"
            "- No markdown (```), no extra explanation\n\n"
            "Text:\n\"\"\"\n{text}\n\"\"\"\n\n"
            "Respond ONLY with a JSON object like this:\n\n"
            "{format_instructions}"
        ),
        input_variables=["text", "topic"],
        partial_variables={"format_instructions": format_instructions}
    )

    final_prompt = prompt.format_prompt(text=text, topic=topic).to_string()

    client = OpenAI(
    base_url = 'https://api.poligpt.upv.es',  
    api_key = 'sk-Icbf-5FyeV0QcLWBC9SNEA',
    timeout=180                    # seconds (global)     
        )

    chat_completion = client.chat.completions.create(
        messages = [
            {'role': 'system', 'content': 'You are an expert in logical reasoning, sustainability reporting, and argument analysis.'},
            {'role': 'user', 'content': final_prompt}
        ],
        model = model_name,
        temperature = 0,
    )

    raw_output = chat_completion.choices[0].message.content
    
    try:
        return pydantic_parser.parse(raw_output)
    except OutputParserException as err:
        print("Parse failed:", err)
        return ArgumentResponse(arguments=[])

# 5. Wrapper function for pipeline
def extract_arguments_from_text(text, topic, model_name) -> List[str]:
    result = extract_arguments_json(text, topic, model_name)
    return result.arguments

# 6. Main document-level processor
def process_document(pages, model_name, topic=""):
    extended_pages = extend_pages_with_next_sentence(pages)
    processed = []
    for page in extended_pages:
        print(f"\n--- Processing Page {page['page']} ---")
        #print("Text to analyze:\n", page["text"])
        
        arguments = extract_arguments_from_text(page["text"], topic, model_name)
        
        print("Extracted Arguments:")
        for i, arg in enumerate(arguments, 1):
            print(f"{i}. {arg}")

        processed.append({
            "page": page["page"],
            "text": page["text"],
            "arguments": arguments
        })
    return processed


# 7. File I/O
def save_to_json(processed, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2, ensure_ascii=False)

def process_directory(input_dir, output_dir, prefix, model_name, topic="", sgd_number=None):
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".json") and filename.startswith(prefix):
            filepath = os.path.join(input_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                pages = json.load(f)

            section_name = filename.replace(".json", "")
            processed = process_document(pages, model_name, topic)

            for item in processed:
                item["section"] = section_name  # Add section identifier
                all_results.append(item)
                
    return all_results



## SGD 1: Poverty

In [23]:
topic = "SGD 1 (Poverty): End poverty in all its forms everywhere"
sgd_number = "1"
resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Extracted Arguments:
1. The disruptions caused by these multiple crises has aggravated fiscal-space issues in low-income countries (LICs) and in lower-middle income countries (LMICs), leading to a reversal in progress on several goals and indicators.
2. To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
3. The Stimulus’ urgent objective is to address the chronic shortfall of international SDG financing confronting the LICs and LMICs, and to ramp up financing flows by at least US$500 billion by 2025.

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:
1. The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
2. At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
3. As called for by United Nations Secretary-

## SGD 2: Hunger

In [24]:
topic = "SGD 2 (Hunger): End hunger, achieve food security and improved nutrition and promote sustainable agriculture"
sgd_number = "2"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:
1. 4. Sustainable ecosystems, sustainable agriculture, and climate resilience: the transition to sustainable land use, healthy diets, and resilience to ongoing climate change;

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 16 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:
1. The 2021 UN Food Systems Summit raised many urgent concerns around improving the sustainability, affordability, and quality of food across the world (SDG 2).
2. Overall, the Food Systems Summit highlighted the need for an integrated and global approach to addressing food systems challenges, including food security, rural development, the reduction of food waste, transparency along the value chain, sustainable diets, and the fight against climate change.

--- Processing Page 1

## SGD 3: Health

In [25]:
topic = "SGD 3 (Health): Ensure healthy lives and promote well-being for all at all ages"
sgd_number = "3"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:
1. Universal health access and coverage: an expansion of health coverage to ensure universal access to both preventative and curative services;

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 16 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:
1. Human capital: The skills and health of a productive citizenry, supported by universal health access and coverage, quality education, shared data and knowledge, promotion of a culture of peace and non-violence, global citizenship, and the appreciation of cultural diversity.
2. Infrastructure: Energy production and distribution, land and sea transport, telecommuni

## SGD 4: Education

In [26]:
topic = "SGD 4 (Education): Ensure inclusive and equitable quality education"
sgd_number = "4"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:
1. 1. Universal quality education and innovation-based economy: a massive increase in investments in quality education and in science and technology innovation systems;
2. 6. Transformation to universal digital access and services: actions by governments at all levels to ensure universal access to digital services including online payments, finance, telemedicine, online education, and others, while ensuring privacy and online safety.

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 16 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:
1. Providing quality education (SDG 4) for all children is perhaps the single most important key to achieving sustainable development in the long term.
2. The UN General Assembly’s Transforming Education Summit held in Se

## SGD 5: Gender

In [27]:
topic = "SGD 5 (Gender): Achieve gender equality and empower all women and girls"
sgd_number = "5"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:
1. This includes equality of opportunities for girls and women (SDG 5)

--- Processing Page 16 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 22 ---
Extracted Arguments:

--- Processing Page 23 ---
Extracted Arguments:

--- Processing Page 24 ---
Extracted Arguments:

--- Processing Page 25 ---
Extracted Arguments:
1. gender equality

--- Processing Page 26 ---
Extracted Arguments:

--- Processing Page 27 ---
Extracted Arguments:

--- Processing Page 28 ---
Extracted Arguments:

--- Processing Page 29 ---
Ex

## SGD 6: Water and sanitation

In [28]:
topic = "SGD 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation for all"
sgd_number = "6"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)




--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 16 ---
Extracted Arguments:
1. Water scarcity affects more than 40% of the world’s population.
2. An estimated 1.8 billion people depend on drinking water contaminated by human waste.
3. Unsustainable water management practices, including chemical discharges into water supply systems for irrigation, affect the functioning of ecosystems services.

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:
1. overuse of both green and blue water

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:
1. public buildings (e.g., schools and hospitals), and safe water and sanitation.

--- Processing Page 21 ---
Extracted Arguments:

--- Processing Page 

## SGD 7: Clean Energy

In [29]:
topic = "SGD 7 (Clean Energy): Ensure access to affordable, reliable, sustainable and modern energy for all"
sgd_number = "7"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:
1. 3. Zero-carbon energy systems: the transition by 2050 of energy systems to net-zero emissions;

--- Processing Page 15 ---
Extracted Arguments:

--- Processing Page 16 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:
1. Infrastructure: Energy production and distribution, land and sea transport, telecommunications, digital information services, public buildings (e.g., schools and hospitals), and safe water and sanitation.

--- Processing Page 21 ---
Extracted Arguments:
1. Challenges such as decarbonization cannot be met with existing technologies alone, and so depend on continued innovation and scientific research, especial

## SGD 8: Decent Work, Economic Growth

In [30]:
topic = "SGD 8 (decent work, economic growth): Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all"
sgd_number = "8"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Extracted Arguments:
1. At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
2. To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
3. The Stimulus’ urgent objective is to address the chronic shortfall of international SDG financing confronting the LICs and LMICs, and to ramp up financing flows by at least US$500 billion by 2025.
4. Greatly increase funding to national and subnational governments and private businesses, especially in LICs and LMICs, to carry out needed SDG investments.
5. Revise the credit rating system and debt sustainability metrics to facilitate long-term sustainable development.
6. Align private business investment flows with the SDGs, through improved national planning, regulation, reporting, and oversight.

--- Processing 

## SGD 9: Infrastructure, industrilization, innovation

In [31]:
topic = "SGD 9 (Infrastructure, industrilization, innovation): Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation"
sgd_number = "9"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 13 ---
Extracted Arguments:

--- Processing Page 14 ---
Extracted Arguments:
1. 1. Universal quality education and innovation-based economy: a massive increase in investments in quality education and in science and technology innovation systems;
2. 5. Sustainable cities: urban infrastructure and services to ensure productive, safe, inclusive, and healthful cities for a world that will be around 70 percent urbanized in 2050;

--- Processing Page 15 ---
Extracted Arguments:
1. Supporting innovation to broaden social inclusion and environmental sustainability;

--- Processing Page 16 ---
Extracted Arguments:

--- Processing Page 17 ---
Extracted Arguments:

--- Processing Page 18 ---
Extracted Arguments:

--- Processing Page 19 ---
Extracted Arguments:

--- Processing Page 20 ---
Extracted Arguments:
1. Infrastructure: Energy production and distribution, land and sea transp

## SGD 10: Inequality

In [33]:
topic = "SGD 10 (Inequality): Reduce inequality within and among countries"
sgd_number = "10"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---


APITimeoutError: Request timed out.

## SGD 11: Sustainable cities

In [ ]:
topic = "SGD 11 (Sustainable Cities, Sustainable Communities): Make cities and human settlements inclusive, safe, resilient and sustainable"
sgd_number = "11"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 12: Responsible Consumption, Responsible Production

In [ ]:
topic = "SGD 12 (Responsible Consumption, Responsible Production): Ensure sustainable consumption and production patterns"
sgd_number = "12"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:
1. Spillover effects stem from cross-border pollution flows, economic and financial flows, multilateralism and security effects, and environmental and social impacts embodied in international trade.

--- Processing Page 5 ---
Extracted Arguments:
1. The latter arises from the globalization of value chains, where products consumed in one country cause social and environmental impacts along their production processes, which are not considered in the consuming country’s national statistics.
2. High income countries, given their consumption levels, are particularly responsible for trade-related spillovers.
3. Accounting for these spillovers becomes crucial for an accurate assessment of contributions towards sustainable development.

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. Integrating CBA int

## SGD 13: Climate change

In [ ]:
topic = "SGD 13 (Climate change): Take urgent action to combat climate change and its impacts"
sgd_number = "13"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:
1. For instance, in some G20 economies (e.g., Germany, France, the UK), offshore GHG emissions are more than 40 percent of their total GHG footprint (Figure 2).

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:
1. Integrating CBA into national statistics and SDG monitoring enhances the visibility of spillover effects and the understanding of global SDG performance.
2. Eurostat’s annual SDGs monitoring report also features a dedicated section on spillover effects, including indicators on CO₂ emissions and cropland footprints of EU members (Eurostat, 2023).
3. Targets for trade-related spillovers should cover environmental and social impacts linked to global value chains, such as GHG emissions, air pollution, resource depletion (e.g., material footprints

## SGD 14: Life bellow water

In [ ]:
topic = "SGD 14 (Life bellow Water): Conserve and sustainably use the oceans, seas and marine resources for sustainable development"
sgd_number = "14"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 15: Life on land

In [ ]:
topic = "SGD 15 (Life on land): Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss"
sgd_number = "15"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:
1. GIS data is used by the World Resources Institute’s Global Forest Watch initiative to track global deforestation and its causes (Curtis et al., 2018).
2. Recognizing that wood consumed by the EU often comes from regions with poor forest management, illegal logging, and degradation of primary forests, VPAs are negotiated by the EU and timber-exporting countries to only allow legally produced imports. Then, timber legality practices, promoting sustainable forestry (EU FLEGT Facility, 2009).

--- Processing Page 9 ---
Extracted Arguments:
1. assurance systems push exporting countries to improve their forest management practices, promoting sustainable fores

## SGD 16: Peace, Justice, Strong Institutions

In [ ]:
topic = "SGD 16 (Peace, Justice, Strong Institutions): Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels"
sgd_number = "16"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:

--- Processing Page 3 ---
Extracted Arguments:

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Arguments:

--- Processing Page 6 ---
Extracted Arguments:

--- Processing Page 7 ---
Extracted Arguments:

--- Processing Page 8 ---
Extracted Arguments:

--- Processing Page 9 ---
Extracted Arguments:

--- Processing Page 10 ---
Extracted Arguments:

--- Processing Page 11 ---
Extracted Arguments:


## SGD 17: Partnerships, sustainable development

In [ ]:
topic = "SGD 17 (Partnerships, sustainable development):Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development"
sgd_number = "17"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:
1. Curbing negative international spillovers requires multilateral solutions, especially across resource-rich and high income consumer countries.
2. Establish inclusive processes that bring together stakeholders to co-develop multilateral policies and partnerships;
3. The G20’s immediate aim should be to build momentum for a successful Summit of the Future by restoring trust through high-level policy discussions on international spillovers, which have the potential to accelerate the SDG progress.

--- Processing Page 3 ---
Extracted Arguments:
1. The G20 is therefore the right forum for overcoming structural obstacles towards the SDGs such as negative spillover effects.
2. These externalities, or spillovers, must be understood, monitored, and managed effectively through actions by the countries causing them and through international partnerships.

--- Processing Page 4 ---
Extracted Arguments:

--- Processing Page 5 ---
Extracted Argument

## SGD 0: Overarching terms

In [ ]:
topic = "SGD Overarching terms: Sustainable Development Goal, SDG, Agenda 2030, leave no one behind, Voluntary National Review, SDG transformations, "
sgd_number = "0"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 2 ---
Extracted Arguments:
1. The G20 is an important international forum to promote the implementation of the 2030 Agenda for Sustainable Development.
2. These are undesirable external economic, social, environmental, and security effects from a country’s actions which undermine other countries’ efforts to achieve the SDGs.
3. Assessments show an inverse relationship between domestic SDG progress and the generation of negative spillovers, with high income countries (many G20 countries) having a better performance on the SDGs domestically (higher SDG Index scores), but also more negative impacts abroad (lower International Spillover Index scores).
4. All countries, and especially the G20 given their combined economic weight, must better understand, monitor and manage their spillovers to allow all countries to meet the SDGs.
5. Curbing negative international spillovers requires multilateral solutions, especially across resource-rich and high income consumer countrie